# Pipeline example


Please make sure the venv is installed and the environment variables are set up as described in the README.md file.

In [ ]:
# Imports
import dotenv
from IPython.display import Markdown, display

dotenv.load_dotenv()

# Extract the text from the pdf


To extract the text from the pdf, we can use any implementation of the `PdfExtractorInterface` class. and pass the pdf data as bytes to the `extract` method.

```python
class PdfExtractorImplementation(PdfExtractorInterface):
    def forward(self, input: bytes) -> str:
        return "Hello, world!"

pdf_extractor = PdfExtractorImplementation()
text = pdf_extractor.forward(pdf_data)
```

These implementations are available in the `llm_synthesis.transformers.pdf_extraction` module.

- `DoclingPDFExtractor`
- `MistralPDFExtractor`


In [ ]:
# declare paper name and id here
# e.g. paper_name = "Adeosun_2024_Direct" without .pdf in the end!
paper_name = "Teng_2024_Ru"
paper_id = "Teng2025Ru"
suppl_name = "Teng_2024_Ru_SI"

# directory of pdfs
DATA_DIR = "../../../data/ammonia_cracking_pdf/"

## MistralPDFExtractor

Extract Data with Mistral.

NB: You need to have a mistral account and a valid API key.


In [ ]:
from llm_synthesis.transformers.pdf_extraction import MistralPDFExtractor

with open(DATA_DIR + paper_name + ".pdf", "rb") as f:
    pdf_data = f.read()

pdf_extractor = MistralPDFExtractor(
    structured=False
)  # You can set the api key as an argument or in the environment variable MISTRAL_API_KEY (use a .env file)
mistral_extracted_text = pdf_extractor.forward(pdf_data)

In [ ]:
display(Markdown(mistral_extracted_text))

Mistral version: Save markdown and create paper object


In [ ]:
# Let's save that for later

with open(DATA_DIR + paper_name + ".md", "w+", errors="replace") as f:
    f.write(mistral_extracted_text)

In [ ]:
from llm_synthesis.transformers.pdf_extraction import MistralPDFExtractor

with open(DATA_DIR + suppl_name + ".pdf", "rb") as f:
    pdf_data = f.read()

with open(DATA_DIR + suppl_name + ".md", "w+", errors="replace") as f:
    f.write(mistral_extracted_text)

pdf_extractor = MistralPDFExtractor(
    structured=False
)  # You can set the api key as an argument or in the environment variable MISTRAL_API_KEY (use a .env file)
mistral_extracted_text_si = pdf_extractor.forward(pdf_data)

In [ ]:
# Let's make it a paper object
with open(DATA_DIR + paper_name + ".md", "w+") as f:
    mistral_extracted_text = f.read()
from llm_synthesis.models.paper import Paper

paper = Paper(
    id=paper_id,
    name=paper_name,
    publication_text=mistral_extracted_text,
    si_text=mistral_extracted_text_si,
)

In [ ]:
paper

# Extract text from a markdown text


These functions are available in the `llm_synthesis.transformers.text_extraction` module. From which you can extract any arbitrary text from the publication text.

We currently use dspy to extract text from a markdown text.

Here are a few examples of how to use it.


In [ ]:
from llm_synthesis.transformers.material_extraction import (
    make_dspy_text_extractor_signature,
)
from llm_synthesis.utils.dspy_utils import get_llm_from_name
from llm_synthesis.utils.markdown_utils import remove_figs

# Let's make a signature for the text extraction
signature = make_dspy_text_extractor_signature(
    signature_name="ExtractSynthesisParagraph",
    instructions="Extract the synthesis paragraph from the markdown publication text.",
    # instructions="Extract all paragraphs describing the synthesis procedure from the markdown publication text. Look for relevant section titles (e.g., “Catalyst Preparation,” “Pd nanoparticles”). Mention all synthesis steps, do not summarize, shorten, or modify the paragraph!",
    input_description="The markdown publication text to extract the synthesis paragraph from.",
    output_name="synthesis_paragraph",
    output_description="The extracted synthesis paragraph.",
)

# Let's first remove the figures from the publication text
paper.publication_text = remove_figs(paper.publication_text)
paper.si_text = remove_figs(paper.si_text)

# Let's make a language model
# lm = get_llm_from_name("gemini-2.0-flash", {"temperature": 0.0})
lm = get_llm_from_name("mistral-small", {"temperature": 0.0})

# collect all materials

XXX

# Extract Structured Data from text

These functions are available in the `llm_synthesis.transformers.structured_data_extraction` module. From which you can extract any arbitrary structured data from the publication text.

We currently use dspy to extract structured data from a markdown text.

Here are a few examples of how to use it.


In [ ]:
from llm_synthesis.transformers.synthesis_extraction import (
    DspySynthesisExtractor,
    make_dspy_synthesis_extractor_signature,
)
from llm_synthesis.utils.dspy_utils import get_llm_from_name

# Let's make a signature for the structured data extraction
signature = make_dspy_synthesis_extractor_signature(
    signature_name="ExtractStructuredSynthesis",
    instructions="Extract the structured synthesis from the synthesis paragraph.",
    input_description="The synthesis paragraph to extract the structured synthesis from.",
    output_name="structured_synthesis",
    output_description="The extracted structured synthesis.",
)

lm = get_llm_from_name("gemini-2.0-flash", {"temperature": 0.0})
# Let's make a structured data extractor
structured_data_extractor = DspySynthesisExtractor(signature, lm)

# Load synthesis paragraph
# with open("../data/txt_papers/docling/test_" + paper_name + ".md") as f:
with open(DATA_DIR + paper_name + ".md") as f:
    synthesis_paragraph = f.read()

# Let's extract the structured data
structured_data = structured_data_extractor.forward(synthesis_paragraph)

structured_data

In [ ]:
from llm_synthesis.utils.markdown_utils import remove_figs

# with open("../data/txt_papers/docling/test_" + paper_name + ".md") as f:
with open(DATA_DIR + paper_name + ".md") as f:
    publication_text = f.read()

publication_text = remove_figs(publication_text)

# Let's extract the structured data
structured_data_from_publication_text = structured_data_extractor.forward(
    publication_text
)

structured_data_from_publication_text

# Extract figures from the publication text

These functions are available in the `llm_synthesis.transformers.figure_extraction` module. From which you can extract any arbitrary figures from the publication text.

We currently expect the pdf_parser to embed figures in the markdown text to be able to extract figures from a markdown text.


In [ ]:
from llm_synthesis.transformers.figure_extraction import (
    FigureExtractorMarkdown,
)

figure_extractor = FigureExtractorMarkdown()

# with open("../data/txt_papers/docling/test_" + paper_name + ".md", errors="replace") as f:
with open(DATA_DIR + paper_name + ".md", errors="replace") as f:
    publication_text = f.read()

figures = figure_extractor.forward(publication_text)

print(f"{len(figures)} figures was found in this paper.")

In [ ]:
import base64
import io

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Loop through each figure in the 'figures' list
for index, figure in enumerate(figures):
    print("=" * 80)

    # Determine if the figure is quantitative
    is_quantitative = (
        "Quantitative" if figure.quantitative else "Not Quantitative"
    )

    # Print figure information
    print(
        f"Figure {index + 1}: {is_quantitative} - Figure Class: {figure.figure_class}"
    )

    # Decode Base64 image data and open it using PIL
    image_data = base64.b64decode(figure.base64_data)
    image_stream = io.BytesIO(image_data)
    image = Image.open(image_stream)

    # Convert image to NumPy array for visualization
    image_array = np.array(image)

    # Plot the image using Matplotlib
    plt.imshow(image_array)
    plt.axis("off")  # Hide axes for better visual appearance
    plt.title(f"{is_quantitative}: {figure.figure_class}")
    plt.show()

In [ ]:
# Let's print the first figure

import base64

from IPython.display import Image

# Convert base64 string to image and display
Image(base64.b64decode(figures[0].base64_data))

In [ ]:
print("alt_text: ", figures[0].alt_text)
print("context_before: ", figures[0].context_before)
print("context_after: ", figures[0].context_after)
print("figure_reference: ", figures[0].figure_reference)
print("position: ", figures[0].position)

# Extract figure descriptions from the publication text

These functions are available in the `llm_synthesis.transformers.figure_description` module. From which you can get figure descriptions from the publication text and figure info.

The current implementation uses dspy to get figure descriptions from the publication text and figure info.


In [ ]:
from llm_synthesis.models.figure import FigureInfoWithPaper
from llm_synthesis.transformers.figure_description import (
    DspyFigureDescriptionExtractor,
    make_dspy_figure_description_extractor_signature,
)
from llm_synthesis.utils.dspy_utils import get_llm_from_name
from llm_synthesis.utils.markdown_utils import remove_figs

# Let's make a signature for the figure description extraction
signature = make_dspy_figure_description_extractor_signature(
    signature_name="DspyFigureDescriptionExtractorSignature",
    instructions="Extract the figure description from the figure.",
    publication_text_description="The publication text to extract the figure description from.",
    si_text_description="The supporting information text to extract the figure description from.",
    figure_base64_description="The base64 encoded image of the figure to extract the description from.",
    caption_context_description="The text context surrounding the figure position including the figure caption and nearby paragraphs that reference this figure.",
    figure_position_info_description="The information about the figure's position in the document (e.g., 'Figure 2', 'Fig. 3a', 'Scheme 1') to help with contextual understanding.",
    figure_description_description="The extracted figure description.",
)

lm = get_llm_from_name("gpt-4o-mini", {"temperature": 0.0})

# with open("../data/txt_papers/docling/test_" + paper_name + ".md", errors="replace") as f:
with open(
    "../data/txt_papers/mistral/test_" + paper_name + ".md", errors="replace"
) as f:
    publication_text = f.read()

publication_text = remove_figs(publication_text)

figure_info_with_paper = FigureInfoWithPaper(
    **figures[2].__dict__,
    paper_text=publication_text,
    si_text="",
)

figure_description_extractor = DspyFigureDescriptionExtractor(signature, lm)

figure_description = figure_description_extractor.forward(
    figure_info_with_paper
)

figure_description